# 0. 라이브러리 호출

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from google.cloud import bigquery

In [2]:
PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

## questionreport (질문 신고(평가))

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.polls_questionreport`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

      id reason                created_at  question_id  user_id
0   4852  그냥 싫어 2023-05-07 12:14:13+00:00           99   894226
1   4971  그냥 싫어 2023-05-07 13:43:20+00:00           99   906185
2   5389  그냥 싫어 2023-05-08 02:16:54+00:00           99   944035
3   7884  그냥 싫어 2023-05-09 14:19:10+00:00           99   981801
4  11094  그냥 싫어 2023-05-11 13:26:01+00:00           99   887923


In [4]:
df.isna().sum()

id             0
reason         0
created_at     0
question_id    0
user_id        0
dtype: int64

In [5]:
df[['reason', 'created_at', 'question_id', 'user_id']].duplicated().sum()

np.int64(0)

In [6]:
df.value_counts('reason')

reason
그냥 싫어                   28446
나랑 맞지 않는 질문인 것 같음        9541
불쾌한 질문 내용                5386
자꾸 같은 내용의 질문 반복          3202
어떻게 이런 생각을? 이 질문 최고!     1821
한 친구가 질문을 반복적으로 보냄       1701
기타                        480
이 질문은 재미없어요               471
불쾌한 내용이 포함되어 있음           250
오타가 있음                     68
선정적이거나 자극적인 질문             58
Name: count, dtype: int64

## 질문

In [8]:
# 전처리 수행 대상 테이블 호출
sql2 = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.polls_question`
"""

# 판다스 데이터프레임으로 변환
df2 = client.query(sql2).to_dataframe()

print(df2.head())

    id                 question_text                created_at
0   99            가장 신비한 매력이 있는 사람은? 2023-03-31 15:22:53+00:00
1  100  "이 사람으로 한 번 살아보고 싶다" 하는 사람은? 2023-03-31 15:22:53+00:00
2  101                     미래의 틱톡커는? 2023-03-31 15:22:54+00:00
3  102               여기서 제일 특이한 친구는? 2023-03-31 15:22:54+00:00
4  103               가장 지켜주고 싶은 사람은? 2023-03-31 15:22:55+00:00


In [9]:
df2.isna().sum()

id               0
question_text    0
created_at       0
dtype: int64

In [14]:
df2[df2[['question_text', 'created_at']].duplicated(keep=False)]

,id,question_text,created_at
1542,1651,vote,2023-06-02 08:06:23+00:00
1543,1652,vote,2023-06-02 08:06:23+00:00
1544,1653,vote,2023-06-02 08:06:23+00:00
1545,1654,vote,2023-06-02 08:06:23+00:00
1546,1655,vote,2023-06-02 08:06:23+00:00
...,...,...,...
4241,4350,vote,2023-06-06 06:15:41+00:00
4438,4547,vote,2023-06-06 06:15:44+00:00
4440,4549,vote,2023-06-06 06:15:44+00:00
4787,4896,vote,2023-06-06 06:15:49+00:00


In [29]:
df2[df2[['question_text', 'created_at']].duplicated()]

,id,question_text,created_at
1543,1652,vote,2023-06-02 08:06:23+00:00
1544,1653,vote,2023-06-02 08:06:23+00:00
1545,1654,vote,2023-06-02 08:06:23+00:00
1546,1655,vote,2023-06-02 08:06:23+00:00
1547,1656,vote,2023-06-02 08:06:23+00:00
1548,1657,vote,2023-06-02 08:06:23+00:00
1549,1658,vote,2023-06-02 08:06:23+00:00
1561,1670,회사 비서에 잘 어울리는 사람,2023-06-02 08:06:23+00:00
1640,1749,vote,2023-06-02 08:06:26+00:00
1664,1773,플래너 잘 쓸 것 같은 사람은?,2023-06-02 08:06:26+00:00


In [30]:
df2[df2['question_text'] == 'vote']

,id,question_text,created_at
87,186,vote,2023-04-01 11:09:15+00:00
384,483,vote,2023-05-02 05:33:11+00:00
540,639,vote,2023-05-11 15:52:44+00:00
587,696,vote,2023-05-15 13:58:24+00:00
603,712,vote,2023-05-15 13:58:30+00:00
616,725,vote,2023-05-15 13:58:35+00:00
627,736,vote,2023-05-15 13:58:40+00:00
698,807,vote,2023-05-15 13:59:11+00:00
772,881,vote,2023-05-15 13:59:44+00:00
831,940,vote,2023-05-15 14:00:10+00:00


In [33]:
display(df2[df2['question_text'] != 'vote']['created_at'].min())
display(df2[df2['question_text'] != 'vote']['created_at'].max())

Timestamp('2023-03-31 15:22:53+0000', tz='UTC')

Timestamp('2023-06-06 06:15:52+0000', tz='UTC')

In [35]:
df2.shape

(5025, 3)

In [28]:
df2[
    df2[['question_text', 'created_at']].duplicated(keep=False)
    & (df2['question_text'] != 'vote')
]

,id,question_text,created_at
1560,1669,회사 비서에 잘 어울리는 사람,2023-06-02 08:06:23+00:00
1561,1670,회사 비서에 잘 어울리는 사람,2023-06-02 08:06:23+00:00
1663,1772,플래너 잘 쓸 것 같은 사람은?,2023-06-02 08:06:26+00:00
1664,1773,플래너 잘 쓸 것 같은 사람은?,2023-06-02 08:06:26+00:00
1820,1929,집안일을 가장 잘 할 것 같은 사람은?,2023-06-02 08:06:32+00:00
1821,1930,집안일을 가장 잘 할 것 같은 사람은?,2023-06-02 08:06:32+00:00
1897,2006,제일 마음씨 착할 것 같은 친구,2023-06-02 08:06:34+00:00
1898,2007,제일 마음씨 착할 것 같은 친구,2023-06-02 08:06:34+00:00
1942,2051,잠버릇이 특이할 것 같은 친구,2023-06-02 08:06:35+00:00
1943,2052,잠버릇이 특이할 것 같은 친구,2023-06-02 08:06:35+00:00


In [42]:
# df[df['question_id'] == 1507]
df[df['question_id'] == 186]


,id,reason,created_at,question_id,user_id
2696,4673,그냥 싫어,2023-05-07 10:15:57+00:00,186,845741
2697,8463,그냥 싫어,2023-05-10 03:23:51+00:00,186,879790
2698,9549,그냥 싫어,2023-05-10 14:24:32+00:00,186,885832
2699,9550,그냥 싫어,2023-05-10 14:24:39+00:00,186,885832
2700,10034,그냥 싫어,2023-05-10 23:37:59+00:00,186,1039156
2701,11515,그냥 싫어,2023-05-11 15:43:39+00:00,186,1054694
2702,11535,그냥 싫어,2023-05-11 15:52:07+00:00,186,1012556
2703,12834,그냥 싫어,2023-05-12 13:12:40+00:00,186,892516
2704,13334,그냥 싫어,2023-05-12 14:28:28+00:00,186,1038236
2705,13704,그냥 싫어,2023-05-12 17:14:12+00:00,186,1032431


In [36]:
question_count = (
    df2[df2['question_text'] != 'vote']
    .drop_duplicates(subset=['question_text', 'created_at'])
    .shape[0]
)

question_count

4952

In [37]:
df2.value_counts('question_text')

question_text
vote                                    56
2세가 가장 귀여울 것 같은 사람은?                     3
눈이 제일 큰 사람은?                             3
같이 밥먹고 싶은 사람은?                           3
지금 뭐하는지 궁금한 친구                           3
                                        ..
개학 하자마자 인싸 될것 같은 사람                      1
너에게 칭찬을 받는다면 가장 기분 좋을 것 같아!              1
사달라는거 다 사줄 것 같은 사람은?                     1
할머니,  할아버지가 돼도 이 친구만큼은 연락하고 지낼 것 같아!     1
개학 하자마자 인사 될것 같은 사람                      1
Name: count, Length: 3903, dtype: int64